# [Baseline] RandomForest → Logistic Regression 개선판 — 학습

KBO 투구 하나가 **제구 성공** 투구일 확률을 예측하는 모델입니다.

- **평가지표**: Brier Skill Score
- 이 노트북은 로지스틱회귀 기반으로, baseline(RandomForest) 대비 다음을 개선했습니다:
  1. 상호작용/비선형 피처 11개 추가
  2. `pitcher_id` 등 익명 ID 컬럼을 실수로 숫자형으로 취급하던 버그 수정 (제외 처리)
  3. `pitcher_hand`/`batter_hand`를 범주형으로 정정 (기존엔 숫자로 잘못 취급됨)
  4. `C` 하이퍼파라미터 로그스케일 전체 탐색으로 정규화 강도 최적화

이 노트북은 모델을 **학습**하여 `./model/lr.pkl` 로 저장합니다.


In [ ]:
import os
import time

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_DIR = "./data"

ID = "row_id"
TARGET = "control_success"

# top_bottom/game_type/base_state는 원래부터 범주형.
# pitcher_hand/batter_hand는 원래 숫자로 잘못 취급되고 있던 걸 여기서 범주형으로 정정.
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_hand", "batter_hand"]

# pitcher_id, batter_id, pitcher_team_id, batter_team_id는 익명 ID일 뿐 크기 의미가 없어서 제외.
# (OneHotEncoder를 쓰기엔 종류가 너무 많고(700명+), asof_* 컬럼들이 이미 선수별 요약 정보를 담고 있어 정보 손실도 적음)
ID_COLS_DROP = ["pitcher_id", "batter_id", "pitcher_team_id", "batter_team_id"]


## 2. 데이터 불러오기

`train.csv` 는 2019~2024 시즌이고 평가 데이터는 2025 시즌입니다.

사용할 피처 목록은 `test.csv` 가 정합니다. `train.csv` 에만 있는 컬럼을 학습에 넣으면
평가 시점에 그 컬럼이 없어 추론이 실패하기 때문입니다.


In [ ]:
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID and c not in ID_COLS_DROP]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])

print("train:", train.shape, "| 피처:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")


train: (1475092, 44) | 피처: 43 (범주형 5, 수치형 38)
시즌: 2019 ~ 2024
제구 성공률: 0.5238


## 3. 피처 엔지니어링 (파생 피처 11개)

로지스틱회귀는 컬럼끼리의 "조합 효과"를 스스로 못 찾기 때문에, 직접 만들어서 추가합니다.
전부 2차(두 컬럼의 곱 또는 제곱) 수준으로, 과적합 위험을 낮게 유지했습니다.

검증(Validation Score) 기준으로 효과가 확인된 것만 남겼습니다
(`pitcher_batter_middle`, `pitcher_recent_trend`, `pitcher_consistency`, `li_pitcher_skill`,
`pitcher_relative_success`, `runner_pressure_skill`은 테스트 결과 효과가 없어 제외).


In [ ]:
train["count_pressure"] = train["strikes_before"] * train["balls_before"]
train["pitcher_batter_success"] = train["asof_pitcher_success_rate"] * train["asof_batter_success_rate"]
train["is_full_count"] = ((train["balls_before"] == 3) & (train["strikes_before"] == 2)).astype(int)
train["pitcher_is_new"] = train["asof_pitcher_success_rate"].isna().astype(int)
train["pitcher_recent5_gap"] = train["asof_pitcher_prev5_game_success_rate"] - train["asof_pitcher_success_rate"]
train["pitcher_success_squared"] = train["asof_pitcher_success_rate"] ** 2
train["fastball_success"] = train["asof_pitcher_fastball_rate"] * train["asof_pitcher_success_rate"]
train["batter_reliability"] = train["asof_batter_success_rate"] * (train["asof_batter_n"] > 10).astype(int)
train["pitcher_reverse_squared"] = train["asof_pitcher_reverse_rate"] ** 2
train["batter_success_squared"] = train["asof_batter_success_rate"] ** 2
train["pitcher_reliability"] = train["asof_pitcher_success_rate"] * (train["asof_pitcher_n"] > 20).astype(int)

new_features = [
    "count_pressure", "pitcher_batter_success", "is_full_count", "pitcher_is_new",
    "pitcher_recent5_gap", "pitcher_success_squared", "fastball_success", "batter_reliability",
    "pitcher_reverse_squared", "batter_success_squared", "pitcher_reliability",
]
FEATURES = FEATURES + new_features
NUM_COLS = NUM_COLS + new_features

print("추가된 파생 피처:", len(new_features), "개")
print("최종 피처 개수:", len(FEATURES))




추가된 파생 피처: 11 개
최종 피처 개수: 54


In [ ]:
trackman = pd.read_csv("data/trackman_history.csv", encoding="utf-8-sig", usecols=["season", "spin_rate", "rel_speed"])
season_stats = trackman.groupby("season")[["spin_rate", "rel_speed"]].mean()

train["league_spin_avg"] = train["season"].map(season_stats["spin_rate"])
train["league_speed_avg"] = train["season"].map(season_stats["rel_speed"])

new_features_trackman = ["league_spin_avg", "league_speed_avg"]
FEATURES = FEATURES + new_features_trackman
NUM_COLS = NUM_COLS + new_features_trackman

print("추가:", new_features_trackman)

추가: ['league_spin_avg', 'league_speed_avg']


## 4. 전처리 정의

범주형(`top_bottom`, `game_type`, `base_state`, `pitcher_hand`, `batter_hand`)은 OneHotEncoder로,
수치형은 결측치 중앙값 대치 + StandardScaler로 처리합니다.


In [ ]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), NUM_COLS),
])


## 5. 모델 정의와 학습

`C=3.16e-06`은 로그스케일 전체 탐색으로 찾은 최적값입니다.

**학습 연도 구성**: leave-one-year-out 테스트 결과, 2022년을 학습에서 제외하는 것이
2024 검증 점수를 크게 높였습니다 (365.13 → 424.93). 2023년은 반드시 포함해야
하고(제외 시 217.99로 급락, 최신 트렌드를 담고 있어 중요), 2022년은 제외하는 것이
더 나은 일반화로 이어지는 것으로 확인되어 학습 데이터에서 제외합니다.


In [ ]:
model = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42, C=3.16e-06)),
])

# 2024 시즌을 검증용으로 떼어 두고, 2022는 leave-one-year-out 테스트 결과에 따라
# 학습에서 제외합니다 (2019, 2020, 2021, 2023 사용).
TRAIN_SEASONS = [2019, 2020, 2021, 2023]
is_val = train["season"] == 2024
is_train = train["season"].isin(TRAIN_SEASONS)
X_train, y_train = train.loc[is_train, FEATURES], train.loc[is_train, TARGET]
X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET]
print("train:", len(X_train), "| val:", len(X_val))

t = time.time()
model.fit(X_train, y_train)
print(f"학습 완료 :: {time.time() - t:.1f}s")


train: 974113 | val: 253507
학습 완료 :: 8.1s


## 6. 계수 확인 (어떤 피처가 유의미한지)

In [ ]:
feature_names = model.named_steps["pre"].get_feature_names_out()
coefs = model.named_steps["clf"].coef_[0]
importance = pd.Series(coefs, index=feature_names).sort_values(key=abs, ascending=False)
print(importance.head(20))


num__pitcher_batter_success                  0.021870
num__asof_pitcher_prev3_game_success_rate    0.020340
num__asof_pitcher_prev5_game_success_rate    0.019927
num__asof_pitcher_reverse_rate              -0.019509
num__pitcher_success_squared                 0.019304
num__asof_pitcher_prev1_game_success_rate    0.018616
num__asof_pitcher_success_rate               0.018376
num__batter_success_squared                  0.017431
num__asof_batter_success_rate                0.016013
cat__game_type_F                             0.014194
cat__game_type_R                            -0.014059
num__league_speed_avg                       -0.012131
num__pitcher_reverse_squared                -0.011916
num__season                                 -0.011505
num__pitcher_reliability                     0.010941
num__batter_reliability                      0.010342
num__asof_pitcher_offspeed_rate              0.006876
num__asof_batter_n                          -0.006585
num__asof_pitcher_ball_rate 

## 7. 검증 — Brier Skill Score

학습 데이터에서 떼어 둔 2024 시즌으로 검증 점수를 계산합니다.


In [ ]:
val_pred = model.predict_proba(X_val)[:, 1]

r = y_val.mean()
brier = ((val_pred - y_val) ** 2).mean()
baseline_brier = r * (1 - r)
score = max(0, 100000 * (1 - brier / baseline_brier))

print(f"Brier: {brier:.6f} | 기준선 r(1-r): {baseline_brier:.6f}")
print(f"Validation Score: {score:.2f}")


Brier: 0.248745 | 기준선 r(1-r): 0.249807
Validation Score: 424.93


## 7-1. Confusion Matrix — TN, FP, FN, TP & F1 Score

이 태스크의 공식 평가지표는 Brier Skill Score지만, 참고용으로 예측 확률을 0.5 기준으로
이진화해 `sklearn.metrics.confusion_matrix` / `f1_score`를 확인합니다.

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

val_pred_label = (val_pred >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, val_pred_label).ravel()
f1 = f1_score(y_val, val_pred_label)

print(f"TN: {tn} | FP: {fp} | FN: {fn} | TP: {tp}")
print(f"F1 Score: {f1:.4f}")

## 8. 전체 데이터로 재학습 & 모델 저장

검증으로 성능을 확인했으니 이제 전체 학습 데이터로 다시 학습합니다.


In [ ]:
# 최종 제출용 모델: 2022를 제외한 전체 데이터(2019, 2020, 2021, 2023, 2024)로 재학습.
# 검증 단계와 다르게 2024도 이제 학습에 포함시킵니다 (더 이상 홀드아웃할 필요 없음).
FINAL_SEASONS = [2019, 2020, 2021, 2023, 2024]
is_final_train = train["season"].isin(FINAL_SEASONS)

t = time.time()
model.fit(train.loc[is_final_train, FEATURES], train.loc[is_final_train, TARGET])
print(f"재학습 완료 :: {time.time() - t:.1f}s (n={is_final_train.sum()})")

os.makedirs("./model", exist_ok=True)
joblib.dump(model, "./model/lr.pkl", compress=3)
print("저장 완료: ./model/lr.pkl")


재학습 완료 :: 7.6s (n=1227620)
저장 완료: ./model/lr.pkl
